-----------
<br><br><a id="0"></a>
# 0. ***Introduction***
---------------------------------
We build a GPT, following the paper "Attention is All You Need" and OpenAI's GPT-2 / GPT-3. We talk about connections to ChatGPT, which has taken the world by storm. We watch GitHub Copilot, itself a GPT, help us write a GPT (meta :D!). We'll utilise the small "[Tiny Shakespeare](https://raw.githubusercontent.com/jcjohnson/torch-rnn/master/data/tiny-shakespeare.txt)" dataset, which contains all of Shakespeare's work in a single file under $1$ MB, instead of a bigger chunk-sized entire internet dataset. This will tremendously reduce our parameter size from the billions. For simplicity and speed, our input tokens will be characters and not words. It's essential to watch the earlier makemore videos to get comfortable with the autoregressive language modeling framework, and basics of tensors & `PyTorch`'s **`torch.nn`**, which we take for granted in this video.

**ChatGPT** is a language model (LM) developed & designed by OpenAI to understand and generate human-like text sequentially based on the input it receives. You can use it for various natural language processing tasks, such as answering questions, having conversations, generating text, and more. For the same input, it provides different outputs when it's rerun numerous times. This shows that it's a probabilistic LM.

<u>Generative Pre-trained Transformer,</u> otherwise known as **GPT**, is a LM that is trained on a siginificant large size of text data to understand and generate human-like text sequentially. The "transformer" part refers to the model's architecture, which was introduced and inspired by the 2017 "[Attention Is All You Need](https://arxiv.org/abs/1706.03762)" paper.

Current implementations from **micrograd (n-grams LM)** to **makemore (MLP, CNN, RNN)** and now **GPT** follow a few key papers:

- Bigram (one character predicts the next one with a lookup table of counts)
- MLP, following [Bengio et al. 2003](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf)
- CNN, following [DeepMind WaveNet 2016](https://arxiv.org/abs/1609.03499) (in progress...)
- RNN, following [Mikolov et al. 2010](https://www.fit.vutbr.cz/research/groups/speech/publi/2010/mikolov_interspeech2010_IS100722.pdf)
  - LSTM, following [Graves et al. 2014](https://arxiv.org/abs/1308.0850)
  - GRU, following [Kyunghyun Cho et al. 2014](https://arxiv.org/abs/1409.1259)
- Transformer, following [Vaswani et al. 2017](https://arxiv.org/abs/1706.03762)

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
import math
import numpy as np

-----------
<br><br><a id="1"></a>
# 1. Baseline Bigram Language Model (LM)
-----------

We establish a simple bigram language model (LM) to get started as our baseline LM. We build our dataset, create our input tokens, split it into train and validation sets, create our bigram LM, train the model and then measure the model performance via cross-entropy loss.


### What is a Bigram?
A **bigram** model looks at **one character** and predicts the **next character**.
"Bi" means two — so bigram = a pair of characters.

**Example from "Hello":**
- H → e
- e → l
- l → l
- l → o

If I show you just **"H"**, can you guess the next letter?
Maybe **"e"** — because "He" is very common in English.
Now if I show you **"He"**, you're even more confident the next letter is **"l"**.
That's exactly what the bigram model does — it learns from all pairs it sees in training data.

### Weakness
The model is **completely blind** to anything before the current character.
It has no memory, no context — which is exactly why we build GPT later.


<a id="101"></a>
## 1.1. Data Reading & Exploration
-----------

Let's download the Tiny Shakespeare dataset, which is about a $1$ MB file, that contains all of Shakespeare's work in one single text file. We read in the text file and, upon inspection, discover it has ~$1$ million characters.

In [2]:
import urllib.request

urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
    'input.txt'
)

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Total characters: {len(text)}")
print()
print(text[:500])

Total characters: 1115394

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [3]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("Vocab Size :", vocab_size)
print("Unique characters in dataset :",''.join(chars))
print("\nTotal number of unique characters in dataset:", vocab_size, '\n')



Vocab Size : 65
Unique characters in dataset : 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz

Total number of unique characters in dataset: 65 



<a id="102"></a>
## 1.2.  Tokenization & Train-Dev Split
-----------


A **tokenizer** is a component used in natural language processing (NLP) to convert raw text of strings into some sequence of integers known as "<u>tokens</u>". An **encoder** allows us turn tokens represented as strings into integers, and a **decoder** allows us to turn our tokens represented as integers back into strings.

We have a very simple character-level tokenizer. There are many different tokenizers, like Google's [SentencePiece](https://github.com/google/sentencepiece) schema (**a subword tokenizer**) or OpenAI's [tiktoken](https://github.com/openai/tiktoken) (**a byte pair encoding, BPE, tokenizer**). These tokenizers operate fundamentally on a sub-word level, which means their vocabulary is much larger (since there are many more permutations of subwords than characters). But the general idea remains the same, we are just turning strings into integers and vice versa.

The large vocabulary size of <u>tiktoken</u>, which is $50257$, enables us to encode a string to a shorter sequence of integers as compared to our own tokenizer of size 65 which generates a longer sequence of integer tokens. The larger the vocabulary size, the shorter the sequence of integer tokens.

So, once we define our encoder and decoder we can then encode our entire dataset. Once we have our encoded dataset, we perform a $90\%:10\%$ train-validation split.

In [4]:
# create a mappong from characters to integers 

stoi = {ch:i for i, ch in enumerate(chars)} # string to integer
itos = {i:ch for i, ch in enumerate(chars)} # integer to string
encode = lambda s: [stoi[c] for c in s]     # encoder: take a string , output a list of integers
decode = lambda l: "".join([itos[i] for i in l])  # decode: take a list of integers, output a string 

print(encode("abdallah"))
print(decode(encode("abdallah")))
print(decode([39, 40, 42, 39, 50, 50, 39, 46]))

[39, 40, 42, 39, 50, 50, 39, 46]
abdallah
abdallah


In [5]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch 

data = torch.tensor(encode(text), dtype= torch.long)
print("size:",data.shape, "\ndtype:",data.dtype)
print(data[:500])

size: torch.Size([1115394]) 
dtype: torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 

In [6]:
# Let's now split up the data into train and validation sets
n = int(0.9 * len(data)) # first 90% will be train, remaining 10% will be val
train_data = data[:n]
val_data = data[n:]

<a id="103"></a>
## 1.3.  Data Loader: Batches
-----------

Let's prepare the model input. We will never feed our model the entire sequence of tokens as prompt at once.
Instead, we will feed it a **randomly drawn but consecutive sequence of tokens.** The model will then predict the next token in the sequence from this prompt.

>We refer to these consecutive, size-limited input sequences of tokens as ***blocks***.
Size-limited means that blocks can have a length of up to `block_size`.

When we sample our dataset, we grab a block of $8$ characters of context plus 1 final character as target. The goal is to learn from the target character during training, predict from the target during evaluation, and generate text from the target during inference.

Suppose we have a `block_size` of $8$, each block actually contains 8 different examples, one for each possible sequence starting with the $1$st initial character. It is important to show our model examples with fewer than `block_size` characters, so that it can learn how to generate text with as little as one character context. Essentially, the transformer should be robust to varying context lengths (1 to `block_size`), which is essential during inference (adequate text generation during sampling with as little as context length of 1 to `block_size`).

In [7]:
block_size = 8 
train_data[:block_size +1 ] # the first 9 char in the data set 

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [8]:
x = train_data[:block_size]     # x = train_data[:8]   --> x = [18, 47, 56, 57, 58,  1, 15, 47]
y = train_data[1:block_size+1]  # y = train_data[1:9]  --> y = [47, 56, 57, 58,  1, 15, 47, 58]
for t in range(block_size): # range (0,1,2,3,4,5,6,7)
    context = x[:t+1]       # x[1] --> x[2] --> x[3] --> x[4] --> x[5] --> ....
    target = y[t]           # y[0] --> y[1] --> ....
    print(f"{t+1}.when input is, {context}, the target: {target}")

1.when input is, tensor([18]), the target: 47
2.when input is, tensor([18, 47]), the target: 56
3.when input is, tensor([18, 47, 56]), the target: 57
4.when input is, tensor([18, 47, 56, 57]), the target: 58
5.when input is, tensor([18, 47, 56, 57, 58]), the target: 1
6.when input is, tensor([18, 47, 56, 57, 58,  1]), the target: 15
7.when input is, tensor([18, 47, 56, 57, 58,  1, 15]), the target: 47
8.when input is, tensor([18, 47, 56, 57, 58,  1, 15, 47]), the target: 58


In [9]:
print('X:', decode(x.tolist()), '  ||  y:', decode(y.tolist()),'\n')
for t in range(block_size):
    context = x[:t+1].tolist()
    target = y[t].tolist()
    print(f"{decode(context)} → {decode([target])}")

X: First Ci   ||  y: irst Cit 

F → i
Fi → r
Fir → s
Firs → t
First →  
First  → C
First C → i
First Ci → t


In the cell above, the representation of X and y is different from our `makemore` version. In makemore, we had a **fixed input context size,** and we padded with `.` in cases where the names were not the full context length. Here, we append each subsequent character step-by-step to ensure the LM learns robustly to **varying context lengths from 1 to `block_size`.**


Now, we feed in the dataset in **batches** of multiple chunks of text that are all stacked up like in a single tensor. This is done for efficiency and speed since GPUs are good at parallel processing/computing. The batches are processed simultaneously and independently of each other.

Since we have `batch_size` 4 and `block_size` 8, one batch will contain a $4\times8$ tensor $X$ and a $4\times8$ tensor $Y$.

* Each row, as a single sample, contains 8 different example contexts, one for each possible sequence starting with the $1$st character until the `block_size`.
* There are 4 rows for the 4 samples in a single batch of `batch_size` 4. Each row has 8 examples, therefore there's a total of 32 training samples.
* Each element in the 4x8 tensor Y contains a single target, each corresponding to one of the 32 examples in X.

In [18]:
print(len(data))

1115394


***torch.stack()*** combines multiple tensors and puts them together along a new dimension.

**Example :**
a = torch.tensor([1, 2, 3]) 
b = torch.tensor([4, 5, 6])

result = torch.stack([a, b])

**Result** : 
tensor([
    [1, 2, 3],
    [4, 5, 6]
])

In [ ]:
torch.manual_seed(1337) 
batch_size = 4  # how many independent sequence will we process in parallet ?
block_size = 8  # what is the maximum context length for predictions ?

def get_batch(split):
    # generate a small batch of data of inputs x and target y
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))  # pick random positon between 0 and  (1115394 - 8, (4,)), assume ix = [10, 100, 500, 1000]
    x = torch.stack([data[i:i+block_size] for i in ix])        # let's assume i = 10, data[10:18], → [A B C D E F G H] || i = 100, data[100:108], → [K L M N O P Q R] ||--> x =[[A B C D E F G H],[K L M N O P Q R]]
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print("input: ")
print("xb.shape :",xb.shape)
print("xb :",xb)
print()
print("target :")
print("yb.shape:", yb.shape)
print("yb :",yb)

print("-----")

for b in range(batch_size):   # batch dimension
    for t in range(block_size): #time dimension
        context = xb[b,:t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target is : {target}")

input: 
xb.shape : torch.Size([4, 8])
xb : tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])

target :
yb.shape: torch.Size([4, 8])
yb : tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
-----
when input is [24] the target is : 43
when input is [24, 43] the target is : 58
when input is [24, 43, 58] the target is : 5
when input is [24, 43, 58, 5] the target is : 57
when input is [24, 43, 58, 5, 57] the target is : 1
when input is [24, 43, 58, 5, 57, 1] the target is : 46
when input is [24, 43, 58, 5, 57, 1, 46] the target is : 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target is : 39
when input is [44] the target is : 53
when input is [44, 53] the target is : 56
when input is [44, 53, 56] the target is : 1
when input is [44, 53, 56, 1] the targ